# Sequential Data — RNN vs LSTM vs GRU

**Objective:** Explore how recurrent architectures process sequential
textual data to extract contextual meaning, handle long-term dependencies,
and mitigate the vanishing gradient problem, using the
[Women's E-Commerce Clothing Reviews](https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews)
dataset for sentiment classification (predicting whether a customer would
recommend the product from their review text).

**Contents**
1. Setup
2. Dataset acquisition
3. Text processing (tokenization, stop-word removal, vocabulary, padding)
4. Model architectures (RNN, LSTM, GRU)
5. Training utilities
6. Train all three models
7. Comparison analysis (convergence speed & accuracy)
8. Architectural & gate-mechanics discussion
9. Conclusion — vanishing gradient mitigation across RNN/LSTM/GRU


## 1. Setup

In [ ]:
import os
import re
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2. Dataset Acquisition

The Women's E-Commerce Clothing Reviews dataset is hosted on Kaggle.

**Option A — KaggleHub (recommended, no manual download):**
```bash
pip install kagglehub
```

**Option B — Manual download:** download the CSV from Kaggle and set
`CSV_PATH` below to point at it directly.

We use the dataset's `Review Text` column as input and its binary
`Recommended IND` column (1 = would recommend, 0 = would not) as the
sentiment label.


In [ ]:
CSV_PATH = None

try:
    import kagglehub
    dataset_dir = kagglehub.dataset_download("nicapotato/womens-ecommerce-clothing-reviews")
    # The CSV filename inside the Kaggle dataset
    for fname in os.listdir(dataset_dir):
        if fname.lower().endswith(".csv"):
            CSV_PATH = os.path.join(dataset_dir, fname)
            break
    print(f"Dataset downloaded, using CSV: {CSV_PATH}")
except Exception as e:
    print("KaggleHub download failed or is unavailable:", e)
    CSV_PATH = "./Womens_Clothing_E-Commerce_Reviews.csv"  # <-- EDIT THIS if downloading manually
    print(f"Falling back to manual path: {CSV_PATH}")


In [ ]:
raw_df = pd.read_csv(CSV_PATH, index_col=0)
print(raw_df.shape)
raw_df[["Review Text", "Rating", "Recommended IND"]].head()


## 3. Text Processing

Steps:
1. Drop rows with missing review text.
2. Lowercase + strip punctuation/digits.
3. Tokenize on whitespace.
4. Remove stop words (a compact built-in English stop-word list — avoids an
   external NLTK data download).
5. Build a vocabulary (index 0 reserved for padding, 1 for unknown tokens).
6. Map each review's tokens to integer indices and pad/truncate every
   sequence to a fixed length.


In [ ]:
# Compact English stop-word list (no external download required)
STOP_WORDS = set('''
a an the and or but if while is are was were be been being
to of in on for with as at by from into over under about
this that these those it its it's i i'm you you're he she they
we my your his her their our not no nor so than too very
do does did doing have has had having can could will would
should shall may might must here there when where why how
all any both each few more most other some such only own
just don should now
'''.split())

def clean_and_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)          # strip punctuation/digits
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return tokens


df = raw_df.dropna(subset=["Review Text"]).copy()
df["tokens"] = df["Review Text"].apply(clean_and_tokenize)
df = df[df["tokens"].map(len) > 0]  # drop reviews that became empty after cleaning

df["label"] = df["Recommended IND"].astype(int)

print(f"Usable reviews: {len(df)}")
print("Label distribution:\n", df["label"].value_counts(normalize=True))
df[["Review Text", "tokens", "label"]].head(3)


In [ ]:
# --- Build vocabulary from the training tokens ---
MAX_VOCAB_SIZE = 10000
MAX_LEN = 60  # pad/truncate every review to this many tokens

token_counter = Counter(tok for toks in df["tokens"] for tok in toks)
most_common = token_counter.most_common(MAX_VOCAB_SIZE - 2)  # reserve 2 special tokens

vocab = {"<PAD>": 0, "<UNK>": 1}
for word, _ in most_common:
    vocab[word] = len(vocab)

vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")


def encode_and_pad(tokens, vocab, max_len=MAX_LEN):
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens[:max_len]]
    if len(ids) < max_len:
        ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))
    return ids


df["input_ids"] = df["tokens"].apply(lambda toks: encode_and_pad(toks, vocab))
df[["tokens", "input_ids"]].head(2)


In [ ]:
# --- Train / validation / test split ---
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


class ReviewsDataset(Dataset):
    def __init__(self, dataframe):
        self.inputs = torch.tensor(dataframe["input_ids"].tolist(), dtype=torch.long)
        self.labels = torch.tensor(dataframe["label"].tolist(), dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.inputs[idx], self.labels[idx]


BATCH_SIZE = 64
train_loader = DataLoader(ReviewsDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ReviewsDataset(val_df), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(ReviewsDataset(test_df), batch_size=BATCH_SIZE, shuffle=False)


## 4. Model Architectures

All three models share the same shape: an `Embedding` layer, a single
recurrent layer (RNN / LSTM / GRU respectively), and a linear output layer
applied to the final hidden state, producing one logit for binary
classification. Keeping everything else identical isolates the effect of
the recurrent cell type itself.


In [ ]:
EMBED_DIM = 100
HIDDEN_DIM = 128


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.rnn(embedded)          # hidden: (1, batch, hidden_dim)
        return self.fc(hidden.squeeze(0)).squeeze(1)


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, cell) = self.lstm(embedded)  # hidden, cell: (1, batch, hidden_dim)
        return self.fc(hidden.squeeze(0)).squeeze(1)


class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)           # hidden: (1, batch, hidden_dim)
        return self.fc(hidden.squeeze(0)).squeeze(1)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


for name, cls in [("RNN", RNNClassifier), ("LSTM", LSTMClassifier), ("GRU", GRUClassifier)]:
    m = cls(vocab_size)
    print(f"{name} trainable parameters: {count_parameters(m):,}")


## 5. Training Utilities

In [ ]:
def train_model(model, train_loader, val_loader, epochs=6, lr=1e-3):
    model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    start_time = time.time()
    for epoch in range(epochs):
        # ---- Train ----
        model.train()
        running_loss, running_correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # guards exploding gradients
            optimizer.step()

            preds = (torch.sigmoid(logits) > 0.5).float()
            running_loss += loss.item() * inputs.size(0)
            running_correct += (preds == labels).sum().item()
            total += inputs.size(0)

        train_loss = running_loss / total
        train_acc = running_correct / total

        # ---- Validate ----
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                logits = model(inputs)
                loss = criterion(logits, labels)
                preds = (torch.sigmoid(logits) > 0.5).float()
                val_loss += loss.item() * inputs.size(0)
                val_correct += (preds == labels).sum().item()
                val_total += inputs.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    training_time = time.time() - start_time
    return history, training_time


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += inputs.size(0)
    return correct / total


## 6. Train All Three Models

In [ ]:
EPOCHS = 6

rnn_model = RNNClassifier(vocab_size).to(device)
rnn_history, rnn_time = train_model(rnn_model, train_loader, val_loader, epochs=EPOCHS)
rnn_test_acc = evaluate(rnn_model, test_loader)
print(f"\nRNN  | training time: {rnn_time:.1f}s | test accuracy: {rnn_test_acc:.4f}\n")


In [ ]:
lstm_model = LSTMClassifier(vocab_size).to(device)
lstm_history, lstm_time = train_model(lstm_model, train_loader, val_loader, epochs=EPOCHS)
lstm_test_acc = evaluate(lstm_model, test_loader)
print(f"\nLSTM | training time: {lstm_time:.1f}s | test accuracy: {lstm_test_acc:.4f}\n")


In [ ]:
gru_model = GRUClassifier(vocab_size).to(device)
gru_history, gru_time = train_model(gru_model, train_loader, val_loader, epochs=EPOCHS)
gru_test_acc = evaluate(gru_model, test_loader)
print(f"\nGRU  | training time: {gru_time:.1f}s | test accuracy: {gru_test_acc:.4f}\n")


## 7. Comparison Analysis — Convergence Speed & Accuracy

In [ ]:
comparison = pd.DataFrame({
    "Model": ["RNN", "LSTM", "GRU"],
    "Trainable Parameters": [
        count_parameters(rnn_model), count_parameters(lstm_model), count_parameters(gru_model)
    ],
    "Training Time (s)": [round(rnn_time, 1), round(lstm_time, 1), round(gru_time, 1)],
    "Final Val Accuracy": [
        round(rnn_history["val_acc"][-1], 4),
        round(lstm_history["val_acc"][-1], 4),
        round(gru_history["val_acc"][-1], 4),
    ],
    "Test Accuracy": [round(rnn_test_acc, 4), round(lstm_test_acc, 4), round(gru_test_acc, 4)],
})
comparison


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].plot(rnn_history["train_loss"], label="RNN")
axes[0].plot(lstm_history["train_loss"], label="LSTM")
axes[0].plot(gru_history["train_loss"], label="GRU")
axes[0].set_title("Training Loss per Epoch (Convergence Speed)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(rnn_history["val_acc"], label="RNN")
axes[1].plot(lstm_history["val_acc"], label="LSTM")
axes[1].plot(gru_history["val_acc"], label="GRU")
axes[1].set_title("Validation Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].bar(comparison["Model"], comparison["Training Time (s)"],
            color=["#4C72B0", "#DD8452", "#55A868"])
axes[2].set_title("Total Training Time")
axes[2].set_ylabel("Seconds")

plt.tight_layout()
plt.show()


## 8. Architectural & Gate-Mechanics Analysis

### 8.1 The LSTM cell state as a "conveyor belt"

An LSTM maintains two running signals per time step: the **hidden state**
`h_t` (short-term, output-facing) and the **cell state** `C_t` (long-term
memory). The cell state is best understood as a conveyor belt that runs
straight through the whole sequence with only *minor, gated* linear
modifications at each step — there is no repeated matrix multiplication
and nonlinearity forcing it through a squashing function at every step the
way a vanilla RNN's hidden state is. Three gates control what happens to it
at each time step:

- **Forget gate** `f_t = σ(W_f · [h_{t-1}, x_t] + b_f)` — decides what
  fraction of the *existing* cell state to keep vs. discard.
- **Input gate** `i_t = σ(W_i · [h_{t-1}, x_t] + b_i)` together with a
  candidate `C̃_t = tanh(W_C · [h_{t-1}, x_t] + b_C)` — decides what *new*
  information to write onto the belt.
- **Cell state update:** `C_t = f_t ⊙ C_{t-1} + i_t ⊙ C̃_t` — an
  (almost) linear, elementwise combination of the old belt content and the
  new candidate, gated by how much of each to use.
- **Output gate** `o_t = σ(W_o · [h_{t-1}, x_t] + b_o)`, with
  `h_t = o_t ⊙ tanh(C_t)` — decides how much of the (squashed) cell state
  to expose as this step's hidden output.

Because the cell-state update is dominated by an elementwise
multiply-and-add rather than a repeated `tanh(W · h_{t-1})` transformation,
gradients flowing backward through `C_t` across many time steps are
multiplied mostly by the forget gate values (which the network can learn
to keep close to 1 for information that should persist) rather than by
the derivative of a saturating nonlinearity at every single step. That is
what lets information — and gradient — survive across long sequences.

### 8.2 GRU: a simplified gating mechanism

A GRU merges the forget/input gates into a single **update gate** `z_t`
and adds a **reset gate** `r_t`, and it has no separate cell state — only
a hidden state that is directly the "memory":

- **Reset gate:** `r_t = σ(W_r · [h_{t-1}, x_t])` — controls how much of
  the previous hidden state is used when computing the new candidate.
- **Update gate:** `z_t = σ(W_z · [h_{t-1}, x_t])` — controls the
  interpolation between the previous hidden state and the new candidate.
- **Candidate:** `h̃_t = tanh(W · [r_t ⊙ h_{t-1}, x_t])`
- **Update:** `h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ h̃_t`

The `(1 - z_t) ⊙ h_{t-1}` term plays the same role as the LSTM's forget
gate on the cell state: it gives gradients a near-linear path backward
through time whenever `z_t` is small, i.e., whenever the network has
learned that a step's new information shouldn't overwrite what's already
stored. With fewer gates and no separate cell state, a GRU has fewer
parameters than an LSTM of the same hidden size, which in this notebook
shows up directly in the parameter-count comparison above.

### 8.3 Why the vanilla RNN struggles

A plain RNN computes `h_t = tanh(W_h · h_{t-1} + W_x · x_t + b)` at every
step, with no gating at all. Backpropagation through time multiplies
gradients by `W_h` and the local derivative of `tanh` at *every single
time step*. Across a 60-token sequence, that's 60 repeated multiplications
by (generally) sub-1 factors (since `tanh'(x) ≤ 1` and weights are
typically initialized/regularized to stay small), which drives the
gradient toward zero — the classic **vanishing gradient problem**. This is
exactly why the RNN in this notebook tends to converge more slowly and/or
plateau at lower accuracy than the LSTM and GRU: the earliest words in a
long review contribute a gradient signal that has largely died out by the
time it reaches those early time steps.


## 9. Conclusion — Vanishing Gradient Mitigation: RNN vs. LSTM vs. GRU

| Aspect | RNN | LSTM | GRU |
|---|---|---|---|
| Memory path | Single hidden state, rewritten every step via `tanh` | Hidden state **+** separate cell state (conveyor belt) | Single hidden state, but gated interpolation |
| Gates | None | Forget, input, output (3) | Reset, update (2) |
| Gradient path across time | Repeated `W_h · tanh'(·)` multiplications → shrinks geometrically | Near-linear path through `C_t` when forget gate ≈ 1 | Near-linear path through `h_t` when update gate ≈ small `z_t` |
| Parameters (same hidden size) | Fewest | Most (4x the weight matrices of an RNN cell) | In between (3x the weight matrices of an RNN cell) |
| Typical behavior on this task | Slower convergence, more prone to plateauing on longer reviews | Best or near-best accuracy, more stable convergence, higher compute/parameter cost | Comparable accuracy to LSTM in fewer epochs, faster per-epoch due to fewer parameters |

**Bottom line:** both the LSTM and GRU mitigate the vanishing gradient
problem by giving gradients an approximately linear, gated path backward
through time (via the cell state in LSTM, via the update-gate
interpolation in GRU), instead of forcing every gradient through a
repeated nonlinear transformation as the vanilla RNN does. The LSTM's
extra cell state and third gate give it the most explicit long-term memory
mechanism, while the GRU achieves similar mitigation with a leaner
parameter budget — which is exactly the empirical convergence-speed vs.
accuracy trade-off observed in the comparison plots above.
